In [5]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

In [6]:
# CSVファイルを読み込み
df = pd.read_csv("../../../input/demo/take5_A.csv")

# 必要な列があるか確認
if not set(['time','x', 'y',]).issubset(df.columns):
    raise ValueError("CSVに time,x, y,列が必要です")


# X-tグラフの描画

## データ抽出

In [7]:
# xの移動距離 = 現在のX座標 - 初期x座標
x_distance = df['x'] - df['x'].iloc[0] # iloc[n]はデータにより可変
y_distance = df['y'] - df['y'].iloc[0]


total_distance  = np.sqrt(x_distance**2 + y_distance**2)
time = df['time']

# 結果をDataFrameにまとめる
total_distance_df= pd.DataFrame({
	'time':time,
	'total_distance':total_distance,
	})

print(total_distance_df)        


      time  total_distance
0    0.305        0.000000
1    0.555        0.170000
2    0.805        0.250000
3    1.054        0.250000
4    1.304        0.330000
..     ...             ...
92  23.303       17.175384
93  23.554       17.063605
94  23.804       16.798039
95  24.053       16.550076
96  24.305       16.478328

[97 rows x 2 columns]


## グラフの描画

In [8]:

# 座標差分と時間差分を計算
dt = df['time'].diff()
dx = df['x'].diff()
dy = df['y'].diff()

# 各点間の移動距離 (ds) = sqrt(dx^2 + dy^2)
ds = np.sqrt(dx**2 + dy**2)

speed = ds / dt

df_speed = pd.DataFrame({
    'time': df['time'].iloc[1:].reset_index(drop=True), 
    'speed': speed.iloc[1:].reset_index(drop=True)
})

In [9]:
# 出力ファイル名
# output_csv = "speedSlowly.csv"

# CSV出力
# df_speed.to_csv(output_csv, index=False)

# 時間も1行目を除いて整形（速度と対応する時間）
speed = df_speed['speed'] 
time = df_speed['time'][1:].reset_index(drop=True)


### 赤い区間の（15.0秒から27.0秒）の平均速度の算出

In [10]:
start_time = 15.0
end_time = 27.0

# 範囲内のデータをフィルタリング
speed_filtered_range = df_speed[(df_speed['time'] >= start_time) & (df_speed['time'] <= end_time)]

# 平均速度を計算
if not speed_filtered_range.empty:
    average_speed_highlight = speed_filtered_range['speed'].mean()
    print(f"\n赤い区間 ({start_time}秒から{end_time}秒) の平均歩行速度: {average_speed_highlight:.4f} m/s")
else:
    print(f"\n赤い区間 ({start_time}秒から{end_time}秒) に対応する速度データがありません。")



赤い区間 (15.0秒から27.0秒) の平均歩行速度: 1.3735 m/s


##  平滑化フィルタをかける


## 移動平均フィルタ

In [11]:
#　ウィンドウの宣言
window_speed=5

df_speed['low_speed']= df_speed['speed'].rolling(window=window_speed).mean()

print(df_speed)
# CSV出力
df_speed.to_csv("speed_copy.csv", index=False)



      time     speed  low_speed
0    0.555  0.680000        NaN
1    0.805  0.320000        NaN
2    1.054  0.000000        NaN
3    1.304  0.320000        NaN
4    1.554  1.000000   0.464000
..     ...       ...        ...
91  23.303  1.874259   1.175147
92  23.554  0.450745   1.065296
93  23.804  1.320000   1.201296
94  24.053  1.054169   1.139038
95  24.305  0.357143   1.011263

[96 rows x 3 columns]
